In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, r2_score
import numpy as np

In [ ]:
df = pd.read_csv("output/behavioral_counts_validation_2025-11-25.csv")

In [ ]:
df['llm_category'] = df['llm_category'].str.replace('"', '')

In [ ]:
len(df.source_pdf.unique())

In [ ]:
df.head()

In [ ]:
print(df["Final_Code"].value_counts())
print(df["Final_Code"].unique())

In [ ]:
print(df["llm_category"].value_counts())
print(df["llm_category"].unique())

In [ ]:
final_to_llm = {
    # Question
    'Q': 'Question', 'Q (reflection only setup)': 'Question', 'Q, CR': 'Question', 'GI Q': 'Question', 
    'SR Q': 'Question', 'NC Q': 'Question', 'NC Q(ruled out SEEK)': 'Question', 'Q (Did-not-meet-the-bar-for SEEK)': 'Question',
    # Reflection Simple
    'SR': 'Reflection Simple', 'SR*': 'Reflection Simple', 'SR; Q': 'Reflection Simple', 'SR GI': 'Reflection Simple',
    'SR Q': 'Reflection Simple', 'SR AF': 'Reflection Simple', 'SR; SEEK': 'Reflection Simple', 'SR; Q': 'Reflection Simple', 'SR; Q (Did-not-reach-bar-for-Seek)': 'Reflection Simple',
    # Reflection Complex
    'CR': 'Reflection Complex', 'CR;AF': 'Reflection Complex', 'CR (+ Emp)': 'Reflection Complex', 'CR +CULTIVATE': 'Reflection Complex', 'CR*': 'Reflection Complex', 'CR; SEEK': 'Reflection Complex', 'SEEK;CR': 'Reflection Complex',
    # Affirm
    'AF': 'Affirm', 'Affirm': 'Affirm', 'AFF': 'Affirm', 'AF SEEK': 'Affirm', 'AF Q': 'Affirm', 'AF Persuade': 'Affirm', 'AFF GI': 'Affirm', 'Seek Affirm': 'Affirm', 'Seek AF': 'Affirm', 'GI Support, NC AF': 'Affirm', 'AF; Seek': 'Affirm',
    # Seeking Collaboration
    'SEEK': 'Seeking Collaboration', 'Seek': 'Seeking Collaboration', 'Seeking Collaboration': 'Seeking Collaboration', 'Seek; GI': 'Seeking Collaboration',
    'GI SEEK': 'Seeking Collaboration', 'Seek; GI': 'Seeking Collaboration', 'Persuade with Seek': 'Seeking Collaboration', 'Seek AF': 'Seeking Collaboration', 'Seek Affirm': 'Seeking Collaboration', 
    'SEEK EMPHASIZE': 'Seeking Collaboration', 'SR; SEEK': 'Seeking Collaboration', 'CR; SEEK': 'Seeking Collaboration', 'GI PWP Seek': 'Seeking Collaboration', 
    'Persuade with Seek': 'Seeking Collaboration', 'SEEK;CR': 'Seeking Collaboration', 'EmpHASIZE': 'Seeking Collaboration', 'Seek': 'Seeking Collaboration', 'EMP': 'Seeking Collaboration', 'Emp 1': 'Seeking Collaboration',
    # Giving Information
    'GI': 'Giving Information', 'GI;': 'Giving Information', 'GI SEEK': 'Giving Information', 'GI Support, NC AF': 'Giving Information', 'GI PERSUADE Q': 'Giving Information', 'GI EMPHASIZE SEEK': 'Giving Information', 'GI PWP Seek': 'Giving Information', 
    'GI Emphasize Seek': 'Giving Information', 'GI; SR': 'Giving Information', 'GI Emphasize Seek': 'Giving Information', 'GI PERSUADE (Q is part of the Persuade)': 'Giving Information',
    # Persuade
    'Persuade': 'Persuade', 'PERSUADE': 'Persuade', 'AF Persuade': 'Persuade', 'GI PERSUADE Q': 'Persuade', 'GI PERSUADE (Q is part of the Persuade)': 'Persuade', 
    'Persuade with': 'Persuade', 'Persuade with Seek': 'Persuade', 'PERSUADE WITH': 'Persuade', 'PERSUADE (could include a GI) Q': 'Persuade', 'Emphasize; Persuade': 'Persuade', 'AFF Persuade': 'Persuade',
    # Persuade with Permission
    'Persuade with': 'Persuade with Permission', 'Persuade with Seek': 'Persuade with Permission', 'PERSUADE WITH': 'Persuade with Permission', 'GI PWP Seek': 'Persuade with Permission', 'Persuade with Permission': 'Persuade with Permission',
    # Emphasizing Autonomy
    'Emphasize': 'Emphasizing Autonomy', 'Emphasize; Persuade': 'Emphasizing Autonomy', 'EMP': 'Emphasizing Autonomy', 'EMP': 'Emphasizing Autonomy', 'EmpHASIZE': 'Emphasizing Autonomy',
    'GI EMPHASIZE SEEK': 'Emphasizing Autonomy', 'Emphasize Q': 'Emphasizing Autonomy', 'EMPHASIZE': 'Emphasizing Autonomy', 'Emphasizing Autonomy': 'Emphasizing Autonomy',
    # Confront
    'Confront': 'Confront', 'CONFRONT': 'Confront', 'CONFRONT warn Q': 'Confront',
    # Not coded / nan / other
    'NOT CODED': '', 'NC': '', 'Structure not coded': '', 'SAME': '', '': '', 'nan': '', '""': '', None: ''
}


In [ ]:
df['final_code_mapped'] = df['Final_Code'].map(final_to_llm)
df['final_code_mapped'] = df['final_code_mapped'].replace('', pd.NA).fillna('No code')
df["llm_category"] = df["llm_category"].replace(["", '""'], pd.NA).fillna('No code')

# Now you can compare df['final_code_mapped'] with df['llm_category']
print(df["final_code_mapped"].value_counts())
print(df["llm_category"].value_counts())


In [ ]:
# Count occurrences for each code (including NaN as "No code" if you want)
final_counts = df['final_code_mapped'].fillna('No code').value_counts()
llm_counts = df['llm_category'].fillna('No code').value_counts()

# Ensure all codes are present for both, reindex and fill missing with zero
all_codes = sorted(set(final_counts.index).union(set(llm_counts.index)))
final_counts = final_counts.reindex(all_codes, fill_value=0)
llm_counts = llm_counts.reindex(all_codes, fill_value=0)

# Make a DataFrame for plotting
plot_df = pd.DataFrame({
    'Human': final_counts,
    'LLM': llm_counts
}).sort_index()

# Plot as horizontal barplot
ax = plot_df.plot(kind='barh', figsize=(10, len(plot_df) * 0.4), color=['#4e79a7', '#f28e2b'])
plt.xlabel('Count')
plt.ylabel('Code')
plt.title('Comparison of Code Counts: Final Code (Human) vs LLM Category')
plt.tight_layout()
plt.show()

In [ ]:
def plot_category(series, xlabel='Count', ylabel='Category', title=None, rotation=0, figsize=(8, 5)):
    plt.figure(figsize=figsize)
    sns.countplot(y=series, order=series.value_counts().index)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if title is None:
        title = f'{ylabel} Distribution'
    plt.title(title)
    plt.yticks(rotation=rotation)
    plt.tight_layout()
    plt.show()

In [ ]:
def normalize_code(cell):
    # Convert to list of lowercase stripped codes
    if pd.isnull(cell): return []
    if isinstance(cell, list): items = cell
    else: 
        # Split by common delimiters, strip spaces, lower
        items = [c.strip().lower() for c in str(cell).replace(';', ',').replace('"', '').replace("'", '').split(',')]
    return sorted([c for c in items if c]) # Remove blanks

# Example DataFrame columns: 'final_code_mapped', 'llm_category'
df['final_codes_norm'] = df['final_code_mapped'].apply(normalize_code)
df['llm_codes_norm'] = df['llm_category'].apply(normalize_code)

# Exact match: both sets must match (i.e. same codes, order irrelevant)
df['exact_match'] = df.apply(lambda x: x['final_codes_norm'] == x['llm_codes_norm'], axis=1)

# Partial overlap: any shared codes, including subset matches
def overlap(row):
    return len(set(row['final_codes_norm']).intersection(set(row['llm_codes_norm']))) > 0

df['partial_overlap'] = df.apply(overlap, axis=1)

# For debugging and analysis, also show intersecting terms and mismatches
df['shared_terms'] = df.apply(lambda x: list(set(x['final_codes_norm']).intersection(set(x['llm_codes_norm']))), axis=1)
df['final_only'] = df.apply(lambda x: list(set(x['final_codes_norm']) - set(x['llm_codes_norm'])), axis=1)
df['llm_only'] = df.apply(lambda x: list(set(x['llm_codes_norm']) - set(x['final_codes_norm'])), axis=1)

# Overview results
print("Exact matches:", df['exact_match'].sum())
print("Partial overlaps:", df['partial_overlap'].sum())
#print(df[['final_code_mapped', 'llm_category', 'exact_match', 'partial_overlap', 'shared_terms', 'final_only', 'llm_only']].head(10))

# Percentage/F1 score etc.
n = len(df)
print(f"Exact match rate: {df['exact_match'].mean():.2%}")
print(f"Partial overlap rate: {df['partial_overlap'].mean():.2%}")

In [ ]:
def reflection_normalize(codes):
    # Check for None, np.nan, empty
    if codes is None:
        return []
    if isinstance(codes, float) and np.isnan(codes):
        return []
    if isinstance(codes, (list, tuple, np.ndarray)):
        if len(codes) == 0:
            return []
        code_list = codes
    else:
        # Assume string -- handle blank
        stripped = str(codes).strip()
        if stripped == '' or stripped.lower() == 'nan':
            return []
        code_list = [c.strip() for c in stripped.replace(';', ',').split(',') if c.strip()]
    out = []
    for c in code_list:
        cl = c.lower()
        if cl in ['reflection simple', 'reflection complex']:
            out.append('reflection')
        elif cl:
            out.append(cl)
    return sorted(set(out))

df['final_codes_norm_ref'] = df['final_codes_norm'].apply(reflection_normalize)
df['llm_codes_norm_ref'] = df['llm_codes_norm'].apply(reflection_normalize)


In [ ]:
# 2) map human codes to canonical behavioral categories
df['human_category'] = df['Final_Code'].map(final_to_llm).fillna('')
df['llm_category'] = df['llm_category'].fillna('')

# 3) per-session (source_pdf) behavior counts for human and LLM
behaviors = [
    'Reflection Simple', 'Reflection Complex', 'Question',
    'Affirm', 'Seeking Collaboration', 'Emphasizing Autonomy',
    'Confront', 'Persuade'
]

human_counts = (
    df
    .groupby(['source_pdf', 'human_category'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=behaviors, fill_value=0)
)

llm_counts = (
    df
    .groupby(['source_pdf', 'llm_category'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=behaviors, fill_value=0)
)

# 4) build metrics per session for human and LLM
metrics = pd.DataFrame(index=human_counts.index)

# --- share of complex reflections: CR / (SR + CR) ---
den_human_ref = (human_counts['Reflection Simple'] +
                 human_counts['Reflection Complex'])
den_llm_ref   = (llm_counts['Reflection Simple'] +
                 llm_counts['Reflection Complex'])

metrics['human_share_complex'] = (
    human_counts['Reflection Complex'] /
    den_human_ref.replace({0: np.nan})
)
metrics['llm_share_complex'] = (
    llm_counts['Reflection Complex'] /
    den_llm_ref.replace({0: np.nan})
)

# --- reflection-to-question ratio: (SR + CR) / Q ---
metrics['human_rq_ratio'] = (
    den_human_ref / human_counts['Question'].replace({0: np.nan})
)
metrics['llm_rq_ratio'] = (
    den_llm_ref / llm_counts['Question'].replace({0: np.nan})
)

# --- total MI-adherent behavior ---
adherent_cats = ['Seeking Collaboration', 'Affirm', 'Emphasizing Autonomy']
metrics['human_mi_adherent'] = human_counts[adherent_cats].sum(axis=1)
metrics['llm_mi_adherent'] = llm_counts[adherent_cats].sum(axis=1)

# --- total MI non-adherent behavior ---
nonadherent_cats = ['Confront', 'Persuade']
metrics['human_mi_nonadherent'] = human_counts[nonadherent_cats].sum(axis=1)
metrics['llm_mi_nonadherent'] = llm_counts[nonadherent_cats].sum(axis=1)

# 5) helper: bias & correlation for a pair of columns
def bias_and_corr(llm_series, human_series):
    # align and drop missing sessions
    tmp = pd.concat([llm_series, human_series], axis=1).dropna()
    llm = tmp.iloc[:, 0]
    hum = tmp.iloc[:, 1]
    
    # bias = mean difference between LLM and human scores
    diff = llm - hum
    bias = diff.mean()
    
    # Pearson correlation between scores
    corr = llm.corr(hum)
    return bias, corr

# 6) compute results (all are BIASES, not shares, except reflection_corr)
results = {}

# share of complex reflections (session-level share, then bias of those shares)
results['share_complex'] = bias_and_corr(
    metrics['llm_share_complex'], metrics['human_share_complex']
)

# reflection-to-question ratio (session-level ratio, then bias of those ratios)
results['rq_ratio'] = bias_and_corr(
    metrics['llm_rq_ratio'], metrics['human_rq_ratio']
)

# total MI-adherent behavior (session-level counts, then bias of counts)
results['mi_adherent'] = bias_and_corr(
    metrics['llm_mi_adherent'], metrics['human_mi_adherent']
)

# total MI non-adherent behavior (session-level counts, then bias of counts)
results['mi_nonadherent'] = bias_and_corr(
    metrics['llm_mi_nonadherent'], metrics['human_mi_nonadherent']
)

# --- Identify reflection categories ---
reflection_cats = ['Reflection Simple', 'Reflection Complex']

# --- Create binary reflection indicators for human and LLM ---
df['human_is_reflection'] = df['human_category'].isin(reflection_cats).astype(int)
df['llm_is_reflection']   = df['llm_category'].isin(reflection_cats).astype(int)

# --- Compute correlation for reflection indicator (no meaningful bias) ---
reflection_corr = df['human_is_reflection'].corr(df['llm_is_reflection'])

# --- Add to results ---
results['reflection_corr'] = reflection_corr

# 7) Create a DataFrame with Category, Bias, Correlation
data = [
    {
        'Category': 'Share Complex Reflections',
        'Bias': results['share_complex'][0],
        'Correlation': results['share_complex'][1]
    },
        {
        'Category': 'Reflection Correlation',
        'Bias': np.nan,  # bias not defined for this measure
        'Correlation': results['reflection_corr']
    },
    {
        'Category': 'Reflection-to-Question Ratio',
        'Bias': results['rq_ratio'][0],
        'Correlation': results['rq_ratio'][1]
    },
    {
        'Category': 'Total MI-Adherent Behavior',
        'Bias': results['mi_adherent'][0],
        'Correlation': results['mi_adherent'][1]
    },
    {
        'Category': 'Total MI Non-Adherent Behavior',
        'Bias': results['mi_nonadherent'][0],
        'Correlation': results['mi_nonadherent'][1]
    }

]

df_results = pd.DataFrame(data)
df_results.to_csv("output/behavioral_scores_validation_results.csv", index=False)


In [ ]:
df

In [ ]:
# --- Identify reflection categories ---
reflection_cats = ['Reflection Simple', 'Reflection Complex']

# --- Create binary reflection indicators for human and LLM ---
df['human_is_reflection'] = df['human_category'].isin(reflection_cats).astype(int)
df['llm_is_reflection']   = df['llm_category'].isin(reflection_cats).astype(int)

# --- Compute correlation ---
reflection_corr = df['human_is_reflection'].corr(df['llm_is_reflection'])

print(f"\nCorrelation: HUMAN reflection vs LLM reflection = {reflection_corr:.3f}")
